# Workshop: Governance & Security

> *"Before go-live: secure RetailHub data with Column Masks, Row Filters, GRANT/REVOKE, and INFORMATION_SCHEMA."*

## Setup

In [0]:
%run ../../setup/00_setup

In [0]:
# Recreate Bronze tables from dataset files (in case they don't exist)
for tbl, fmt, path in [
    ("customers", "csv", f"{DATASET_PATH}/customers/customers.csv"),
    ("orders", "json", f"{DATASET_PATH}/orders/orders_batch.json"),
    ("products", "csv", f"{DATASET_PATH}/products/products.csv"),
]:
    full_name = f"{CATALOG}.{BRONZE_SCHEMA}.{tbl}"
    if not spark.catalog.tableExists(full_name):
        reader = spark.read.format(fmt)
        if fmt == "csv":
            reader = reader.option("header", "true").option("inferSchema", "true")
        reader.load(path).write.mode("overwrite").saveAsTable(full_name)
        print(f"[RECREATED] {full_name}")
    else:
        print(f"[OK] {full_name} exists")

In [0]:
# Ensure Silver tables exist (create from Bronze if missing)
silver_customers = f"{CATALOG}.{SILVER_SCHEMA}.customers"
silver_orders = f"{CATALOG}.{SILVER_SCHEMA}.orders"

if not spark.catalog.tableExists(silver_customers):
    spark.sql(f"""
        CREATE TABLE {silver_customers} AS
        SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.customers
    """)
    print(f"Created {silver_customers} from Bronze")

if not spark.catalog.tableExists(silver_orders):
    spark.sql(f"""
        CREATE TABLE {silver_orders} AS
        SELECT *,
            CASE 
                WHEN CAST(REGEXP_EXTRACT(store_id, '(\\\\d+)') AS INT) <= 10 THEN 'East'
                WHEN CAST(REGEXP_EXTRACT(store_id, '(\\\\d+)') AS INT) <= 20 THEN 'West'
                ELSE 'Central'
            END AS store_region
        FROM {CATALOG}.{BRONZE_SCHEMA}.orders
    """)
    print(f"Created {silver_orders} from Bronze (with store_region)")

df_customers = spark.table(silver_customers)
df_orders = spark.table(silver_orders)
print(f"Customers: {df_customers.count()}, Orders: {df_orders.count()}")

---
## Task 1: Grant Privileges

Grant `SELECT` on the Silver schema to the `analysts` group.

Hint: You need `USE CATALOG`, `USE SCHEMA`, and `SELECT` grants.

In [0]:
# TODO: Grant the three privileges needed for analysts to query silver tables
# - USE CATALOG on the catalog (lets them "enter" the catalog)
# - USE SCHEMA on the silver schema (lets them "enter" the schema)
# - SELECT on the silver schema (lets them read tables)

spark.sql(f"GRANT ... ON CATALOG {CATALOG} TO `analysts`")
spark.sql(f"GRANT ... ON SCHEMA {CATALOG}.{SILVER_SCHEMA} TO `analysts`")
spark.sql(f"GRANT ... ON SCHEMA {CATALOG}.{SILVER_SCHEMA} TO `analysts`")

In [0]:
# Verification
grants_df = spark.sql(f"SHOW GRANTS ON SCHEMA {CATALOG}.{SILVER_SCHEMA}")
grants_df.display()

assert grants_df.filter("Principal = 'analysts'").count() >= 2, "Grants not found for analysts group"
print("Task 1 PASSED")

---
## Task 2: Query INFORMATION_SCHEMA

List all tables in your catalog using `INFORMATION_SCHEMA.TABLES`.

Hint: Filter by `table_schema` to see only Silver tables.

In [0]:
# TODO: Query INFORMATION_SCHEMA to list tables in the Silver schema
# System table: {CATALOG}.INFORMATION_SCHEMA.TABLES
# Columns available: table_schema, table_name, table_type

tables_df = spark.sql(f"""
    SELECT table_schema, table_name, table_type
    FROM {CATALOG}.INFORMATION_SCHEMA....
    WHERE table_schema = '{SILVER_SCHEMA}'
""")
tables_df.display()

In [0]:
# Verification
assert tables_df.count() > 0, "No tables found in Silver schema"
print(f"Found {tables_df.count()} tables in Silver schema")
print("Task 2 PASSED")

---
## Task 3: Create a Column Mask Function

Create a SQL function that masks email addresses. Non-admin users should see only the first 2 characters + `***@***.***`.

Hint: Use `is_account_group_member('admins')` and `CONCAT(LEFT(email, 2), '***@***.***')`.

In [0]:
# TODO: Create an email masking function
# - If the user is a member of "admins" group -> show full email
# - Otherwise -> mask email as "ab***@***.***"
# Use is_account_group_member(group_name) to check group membership

spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{SILVER_SCHEMA}.mask_email(email STRING)
    RETURNS STRING
    RETURN CASE
        -- TODO: condition that checks if user is in "admins" group
        WHEN ... THEN email
        ELSE CONCAT(LEFT(email, 2), '***@***.***')
    END
""")
print("Masking function created")

In [0]:
# Verification -- test the function
test_df = spark.sql(f"SELECT {CATALOG}.{SILVER_SCHEMA}.mask_email('john.doe@example.com') AS masked")
test_df.display()
print("Task 3 PASSED")

---
## Task 4: Apply Column Mask to Table

Apply the `mask_email` function to the `email` column of the `customers` table.

Hint: `ALTER TABLE ... ALTER COLUMN ... SET MASK ...`

In [0]:
# TODO: Apply the mask_email masking function to the email column
# Syntax: ALTER TABLE ... ALTER COLUMN col SET MASK function_name
# The keywords between SET and the function name are: MASK

spark.sql(f"""
    ALTER TABLE {CATALOG}.{SILVER_SCHEMA}.customers
    ALTER COLUMN email SET ... {CATALOG}.{SILVER_SCHEMA}....
""")
print("Column mask applied to email column")

In [0]:
# Verification -- query the table
masked_df = spark.sql(f"SELECT customer_id, email FROM {CATALOG}.{SILVER_SCHEMA}.customers LIMIT 5")
masked_df.display()
print("Task 4 PASSED -- check if email is masked for non-admin users")

---
## Task 5: Create a Row Filter Function

Create a function that restricts visibility of orders by `store_region`. Only users in the matching group can see rows for that region. Admins see all.

Hint: Use `is_account_group_member()` with multiple OR conditions.

In [0]:
# TODO: Create a row-level security filter function
# - admins: see all rows
# - us_team: see only rows where region = 'US'
# - eu_team: see only rows where region = 'EU'
# Use is_account_group_member(group) for group checks

spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{SILVER_SCHEMA}.region_filter(region STRING)
    RETURNS BOOLEAN
    RETURN (
        is_account_group_member('admins')
        OR (is_account_group_member('...') AND region = '...')
        OR (is_account_group_member('...') AND region = '...')
    )
""")
print("Row filter function created")

---
## Task 6: Apply Row Filter to Table

Apply the `region_filter` function to the `orders` table.

Hint: `ALTER TABLE ... SET ROW FILTER ... ON (column)`

In [0]:
# TODO: Apply the region_filter row filter to the orders table
# Syntax: ALTER TABLE ... SET ROW FILTER function_name ON (column_name)
# The column to filter on is "region"

spark.sql(f"""
    ALTER TABLE {CATALOG}.{SILVER_SCHEMA}.orders
    SET ... {CATALOG}.{SILVER_SCHEMA}.region_filter ON (...)
""")
print("Row filter applied to orders table")

In [0]:
# Verification -- check row count (admins should see all rows)
filtered_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {CATALOG}.{SILVER_SCHEMA}.orders").first()["cnt"]
print(f"Visible rows: {filtered_count}")
print("Task 6 PASSED")

---
## Task 7: Query Table Privileges

Use `INFORMATION_SCHEMA.TABLE_PRIVILEGES` to verify who has access to what.

In [0]:
# TODO: Query INFORMATION_SCHEMA to see all privileges granted on tables
# System table: {CATALOG}.INFORMATION_SCHEMA.TABLE_PRIVILEGES
# Columns: grantor, grantee, table_schema, table_name, privilege_type

privs_df = spark.sql(f"""
    SELECT grantor, grantee, table_schema, table_name, privilege_type
    FROM {CATALOG}.INFORMATION_SCHEMA....
    ORDER BY grantee, table_name
""")
privs_df.display()

In [0]:
# Verification
assert privs_df.count() > 0, "No privileges found"
print(f"Found {privs_df.count()} privilege entries")
print("Task 7 PASSED")

---
## Task 8: Remove Mask and Filter (Cleanup)

Remove the column mask and row filter applied earlier.

Hint: `ALTER TABLE ... ALTER COLUMN ... DROP MASK` and `ALTER TABLE ... DROP ROW FILTER`

In [0]:
# TODO: Remove the column mask from email
# Syntax: ALTER TABLE ... ALTER COLUMN col DROP MASK
spark.sql(f"""
    ALTER TABLE {CATALOG}.{SILVER_SCHEMA}.customers
    ALTER COLUMN email ... ...
""")

# TODO: Remove the row filter from orders
# Syntax: ALTER TABLE ... DROP ROW FILTER
spark.sql(f"""
    ALTER TABLE {CATALOG}.{SILVER_SCHEMA}.orders
    ... ... ...
""")

print("Cleanup complete -- mask and filter removed")

In [0]:
# Final verification
clean_df = spark.sql(f"SELECT customer_id, email FROM {CATALOG}.{SILVER_SCHEMA}.customers LIMIT 3")
clean_df.display()
print("Task 8 PASSED -- all governance controls removed")

---

## Task 9: DENY Privilege

DENY blocks access explicitly — overrides any GRANT on the same object.

**Scenario:** The `contractors` principal was mistakenly GRANTed `SELECT` on the Bronze schema.
Use DENY to block access to the `orders` table specifically.

**Expected syntax:**
```sql
DENY SELECT ON TABLE <catalog>.<schema>.<table> TO `<principal>`
```

In [0]:
# TODO: DENY SELECT on orders table to contractors
# - Use DENY on TABLE level
# - Principal: `contractors` (must already exist as a group)
# Uncomment and complete:

# spark.sql(f"""
#     DENY SELECT ON TABLE {CATALOG}.{BRONZE_SCHEMA}.orders TO `contractors`
# """)

# Verify: SHOW GRANTS should include a DENY entry
# spark.sql(f"SHOW GRANTS ON TABLE {CATALOG}.{BRONZE_SCHEMA}.orders").display()

print("Task 9: DENY takes precedence over GRANT. DENY applied at TABLE level.")

In [0]:
# Verification: SHOW GRANTS includes both GRANT and DENY entries
grants = spark.sql(f"""SHOW GRANTS ON TABLE {CATALOG}.{BRONZE_SCHEMA}.orders""") \
              .filter("privilege_type LIKE '%DENY%' OR privilege_type = 'SELECT'")
grants.display()
print("Task 9 complete: DENY verified via SHOW GRANTS")

---
## Summary

| Task | Concept | Key SQL |
|------|---------|--------|
| 1 | Privileges | `GRANT SELECT ON SCHEMA ... TO ...` |
| 2 | Metadata | `INFORMATION_SCHEMA.TABLES` |
| 3-4 | Column Mask | `ALTER TABLE ... ALTER COLUMN ... SET MASK fn` |
| 5-6 | Row Filter | `ALTER TABLE ... SET ROW FILTER fn ON (col)` |
| 7 | Audit | `INFORMATION_SCHEMA.TABLE_PRIVILEGES` |
| 8 | Cleanup | `DROP MASK`, `DROP ROW FILTER` |

← [10 — Governance & Security](../demo/10_governance_and_security.ipynb) | **[ README](../../../README.md)** | [11 — Exam Preparation →](../demo/11_exam_preparation.ipynb)